# GroupDNA: WhatsApp Group Chat Analyzer

**Name:** Umar Suhail  
**Roll number:** Not issued  
**Batch:** Data Science Program November 2026 Batch  
**Date:** 23 November 2026

This notebook analyzes the supplied synthetic `hostel_bois.txt` WhatsApp export and prints a group activity and personality report. It uses Python fundamentals and NumPy, without pandas, plotting libraries, regex, `Counter`, or chat-analysis packages.

> **AI assistance disclosure:** AI-assisted drafting was used for portions of this notebook. Review, understand, and adapt the code and explanations in your own style before submission, as required by the brief.

## Feature 1: Read and parse the chat export
Upload `hostel_bois.txt` to the Colab session, or place it beside this notebook. The parser retains user-sent media and deleted-message records for activity counts while excluding them from word and message-length analysis. System events are counted separately.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
from datetime import datetime, timedelta
import numpy as np

FILE_PATH = 'hostel_bois.txt'
DATE_FORMAT = '%d/%m/%y, %H:%M'
MEDIA_TEXT = '<Media omitted>'
DELETED_TEXT = 'This message was deleted'

def looks_like_timestamp(line):
    # The export starts a new record with DD/MM/YY, HH:MM.
    return len(line) >= 17 and line[2:3] == '/' and line[5:6] == '/' and line[8:10] == ', '

def parse_chat(path):
    messages = []
    system_count = 0
    with open(path, 'r', encoding='utf-8') as chat_file:
        lines = chat_file.read().splitlines()

    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            continue
        if not looks_like_timestamp(line):
            # A continuation line belongs to the previous record when possible.
            if messages:
                messages[-1]['text'] += ' ' + line
            continue
        try:
            timestamp_text, body = line.split(' - ', 1)
            timestamp = datetime.strptime(timestamp_text, DATE_FORMAT)
        except (ValueError, IndexError):
            system_count += 1
            continue
        if ': ' not in body:
            system_count += 1
            continue
        sender, message_text = body.split(': ', 1)
        if not sender.strip():
            system_count += 1
            continue
        messages.append({'timestamp': timestamp, 'timestamp_text': timestamp_text,
                         'sender': sender.strip(), 'text': message_text.strip()})
    return messages, system_count

messages, system_messages = parse_chat(FILE_PATH)
participants = sorted({m['sender'] for m in messages})
print(f'Parsed {len(messages):,} user messages from {len(participants)} participants.')
print(f'System messages skipped: {system_messages}')
print('First parsed records:', messages[:2])
print('Last parsed records:', messages[-2:])

## Feature 2: Group overview and per-person counts
Media and deleted entries count as messages sent. The separate counters below show those special records; content-based metrics use only regular text messages.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
message_counts = {person: 0 for person in participants}
media_counts = {person: 0 for person in participants}
deleted_counts = {person: 0 for person in participants}
text_messages = []
for message in messages:
    person = message['sender']
    message_counts[person] += 1
    if message['text'] == MEDIA_TEXT:
        media_counts[person] += 1
    elif message['text'] == DELETED_TEXT:
        deleted_counts[person] += 1
    else:
        text_messages.append(message)
participants = sorted(participants, key=lambda person: message_counts[person], reverse=True)

first_time = min(m['timestamp'] for m in messages)
last_time = max(m['timestamp'] for m in messages)
first_day = first_time.date()
last_day = last_time.date()
period_days = (last_day - first_day).days + 1
total_messages = len(messages)
print('=' * 60)
print('GROUP OVERVIEW')
print('=' * 60)
print(f'Group           : Hostel Bois 4ever')
print(f'Period          : {first_time:%d %B %Y} to {last_time:%d %B %Y} ({period_days} days)')
print(f'Total messages  : {total_messages:,}')
print(f'Participants    : {len(participants)}')
print(f'System events   : {system_messages}')
print(f'Media messages  : {sum(media_counts.values())}')
print(f'Deleted messages: {sum(deleted_counts.values())}')
print('\nMESSAGES PER PERSON')
for person, count in sorted(message_counts.items(), key=lambda item: item[1], reverse=True):
    percent = count / total_messages * 100 if total_messages else 0
    print(f'  {person:<10}: {count:>4} ({percent:>5.1f}%)')
print('\nCONTENT LENGTH PER PERSON')
for person in participants:
    lengths = [len(m['text'].split()) for m in text_messages if m['sender'] == person]
    total_words = sum(lengths)
    average_words = total_words / len(lengths) if lengths else 0
    print(f'  {person:<10}: {total_words:>5} words; {average_words:.1f} words/message')

## Feature 3: Busiest day and hour

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
day_counts = {}
hour_counts = {hour: 0 for hour in range(24)}
for message in messages:
    day = message['timestamp'].date()
    hour = message['timestamp'].hour
    day_counts[day] = day_counts.get(day, 0) + 1
    hour_counts[hour] += 1
busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)
print('BUSIEST ACTIVITY')
print(f'  Day : {busiest_day:%d %B %Y} ({day_counts[busiest_day]} messages)')
print(f'  Hour: {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 ({hour_counts[busiest_hour]} messages overall)')

## Feature 4: NumPy activity heatmap
Each row is a participant and each column is one hour of the day. Shading is scaled to that participant's busiest hour.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
person_index = {person: index for index, person in enumerate(participants)}
activity = np.zeros((len(participants), 24), dtype=int)
for message in messages:
    activity[person_index[message['sender']], message['timestamp'].hour] += 1

print('ACTIVITY HEATMAP (messages by hour)')
print('          ' + ' '.join(f'{hour:02d}' for hour in range(24)))
for row_index, person in enumerate(participants):
    row = activity[row_index]
    peak = int(np.max(row))
    shades = []
    for value in row:
        ratio = value / peak if peak else 0
        shades.append('█' if ratio >= .75 else '▒' if ratio >= .50 else '░' if ratio > 0 else '.')
    print(f'{person:<9} ' + ' '.join(shades))

## Feature 5: Top words
Tokenization uses string methods and a hand-built dictionary counter. The simple punctuation cleanup is intentional so the code stays within the assignment constraints.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
stop_words = {'i','is','the','a','an','and','or','to','of','in','on','for','it','you','we','are','was','this','that','my','me','be','but','with','have','has','do','did','at','so','as','from','what','who','if','not'} | {'how','about','hai','am','today','he','his','just','which','everyone','telling','up','one','had','started','they','no','then','when','because','been','being','were','there','their','them','some','into','out','back','really','finally'}
punctuation = ".,!?;:\"()[]{}<>-_"
word_counts = {}
person_word_counts = {person: {} for person in participants}
for message in text_messages:
    for raw_word in message['text'].lower().split():
        word = raw_word.strip(punctuation)
        if word and word not in stop_words:
            word_counts[word] = word_counts.get(word, 0) + 1
            person = message['sender']
            person_word_counts[person][word] = person_word_counts[person].get(word, 0) + 1

print("THIS GROUP'S FAVOURITE WORDS")
for word, count in sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:10]:
    bar = '█' * max(1, round(count / max(word_counts.values()) * 20))
    print(f'  {word:<12} {bar:<20} {count}')
print('\nTOP WORDS PER PERSON')
for person in participants:
    top = sorted(person_word_counts[person].items(), key=lambda item: item[1], reverse=True)[:5]
    print(f'  {person:<10}: ' + ', '.join(f'{word} ({count})' for word, count in top))

## Feature 6: Response patterns and silent streaks
A response is a message preceded immediately by another participant's message in timestamp order. Its delay is measured from that preceding message. Silent days include dates from the group's first to last active date.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
ordered = sorted(messages, key=lambda item: item['timestamp'])
response_gaps = {person: [] for person in participants}
for previous, current in zip(ordered, ordered[1:]):
    if previous['sender'] != current['sender']:
        gap_minutes = (current['timestamp'] - previous['timestamp']).total_seconds() / 60
        response_gaps[current['sender']].append(gap_minutes)

average_response = {}
for person, gaps in response_gaps.items():
    if gaps:
        average_response[person] = sum(gaps) / len(gaps)

all_days = [first_day + timedelta(days=offset) for offset in range(period_days)]
active_days = {person: set() for person in participants}
for message in messages:
    active_days[message['sender']].add(message['timestamp'].date())
longest_silent = {}
for person in participants:
    best = current_streak = 0
    for day in all_days:
        if day not in active_days[person]:
            current_streak += 1
            best = max(best, current_streak)
        else:
            current_streak = 0
    longest_silent[person] = best

print('RESPONSE PATTERNS')
if average_response:
    fastest = min(average_response, key=average_response.get)
    slowest = max(average_response, key=average_response.get)
    print(f'  Fastest replier: {fastest} ({average_response[fastest]:.1f} minutes average)')
    print(f'  Slowest replier: {slowest} ({average_response[slowest] / 60:.1f} hours average)')
print('\nLONGEST SILENT STREAKS')
for person in sorted(participants, key=lambda name: longest_silent[name], reverse=True):
    print(f'  {person:<10}: {longest_silent[person]} days')

## Feature 7: Personality archetypes
The six dataset-specific archetypes are assigned first using the quantitative signals in the brief. To keep assignments exclusive, each selected person is removed from the available pool; if a rule's qualifying signal is absent, the strongest remaining candidate is used. The two fallback archetypes then fill any remaining participants. Ties follow alphabetical participant order. The scored signals are printed so assignments can be reviewed.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
def message_bursts(person):
    lengths = []
    run = 0
    for message in ordered:
        if message['sender'] == person:
            run += 1
        elif run:
            lengths.append(run)
            run = 0
    if run:
        lengths.append(run)
    return sum(lengths) / len(lengths) if lengths else 0

def content_stats(person):
    own = [m['text'] for m in text_messages if m['sender'] == person]
    count = message_counts[person]
    word_total = sum(len(text.split()) for text in own)
    night_count = sum(1 for m in messages if m['sender'] == person and (m['timestamp'].hour >= 23 or m['timestamp'].hour <= 4))
    drama_count = sum(1 for text in own if (len(text) >= 3 and text.isupper()) or text.count('!') >= 2)
    caring_terms = ['okay','safe','eat','sleep','take care','are you','please','reminder','drink water',"don't forget"]
    caring_count = sum(text.lower().count(term) for text in own for term in caring_terms)
    humor_terms = ['lol','lmao','haha','rofl','lmfao']
    humor_count = sum(text.lower().count(term) for text in own for term in humor_terms)
    question_count = sum(1 for text in own if text.rstrip().endswith('?'))
    return {'avg_burst': message_bursts(person), 'avg_words': word_total / len(own) if own else 0,
            'night_pct': night_count / count if count else 0,
            'drama_pct': drama_count / len(own) if own else 0,
            'caring': caring_count, 'humor_pct': humor_count / count if count else 0,
            'question_pct': question_count / count if count else 0,
            'silent_pct': (period_days - len(active_days[person])) / period_days if period_days else 0}

stats = {person: content_stats(person) for person in participants}
# Dataset-specific qualifying rules from the brief; global uniqueness prevents duplicate labels.
rule_specs = [
    ('THE SPAMMER', 'avg_burst', 3, 'max'),
    ('THE GROUP MOM', 'caring', 0, 'max'),
    ('THE NIGHT OWL', 'night_pct', .60, 'max'),
    ('THE STORYTELLER', 'avg_words', 30, 'max'),
    ('THE DRAMA QUEEN', 'drama_pct', .30, 'max'),
    ('THE GHOST', 'silent_pct', .60, 'max'),
    ('THE COMEDIAN', 'humor_pct', 0, 'max'),
    ('THE QUESTION MASTER', 'question_pct', .25, 'max')]
remaining = set(participants)
archetypes = {}
for label, metric, threshold, direction in rule_specs:
    eligible = [p for p in remaining if stats[p][metric] > threshold]
    if eligible:
        winner = max(eligible, key=lambda p: (stats[p][metric], p))
        archetypes[winner] = (label, metric, stats[winner][metric])
        remaining.remove(winner)
# Any participants left get the best remaining fallback signal.
for person in sorted(remaining):
    fallback = max(('THE COMEDIAN', 'humor_pct'), ('THE QUESTION MASTER', 'question_pct'),
                   key=lambda item: stats[person][item[1]])
    archetypes[person] = (fallback[0], fallback[1], stats[person][fallback[1]])

print('PERSONALITY ARCHETYPES')
for person in participants:
    label, metric, score = archetypes[person]
    if metric.endswith('_pct'):
        detail = f'{score:.1%}'
    else:
        detail = f'{score:.1f}'
    print(f'  {person:<10} -> {label:<20} ({metric}: {detail})')

print('\nSPECIAL MESSAGE COUNTS')
for person in participants:
    print(f'  {person:<10}: {media_counts[person]} media, {deleted_counts[person]} deleted')

## Feature 8: Final GroupDNA report
This final cell brings the main findings together in a compact, screenshot-friendly report.

In [ ]:
# AI-assisted: reviewed and adapted with AI help.
print('\n' + '=' * 68)
print('  GROUPDNA REPORT - Hostel Bois 4ever')
print(f'  {period_days} days | {total_messages:,} messages | {len(participants)} members')
print(f'  System: {system_messages} | media: {sum(media_counts.values())} | deleted: {sum(deleted_counts.values())}')
print('=' * 68)
print(f'  Period          : {first_time:%d %B %Y} to {last_time:%d %B %Y}')
print(f'  Busiest day     : {busiest_day:%d %B %Y} ({day_counts[busiest_day]} messages)')
print(f'  Busiest hour    : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00')
print('\n  MESSAGES PER PERSON')
max_count = max(message_counts.values()) if message_counts else 1
for person, count in sorted(message_counts.items(), key=lambda item: item[1], reverse=True):
    bar = '█' * max(1, round(count / max_count * 20))
    print(f'  {person:<10} {bar:<20} {count:>4} ({count / total_messages * 100:4.1f}%)')
print('\n  ACTIVITY HEATMAP (00-23 hours)')
print('          ' + ' '.join(f'{hour:02d}' for hour in range(24)))
for row_index, person in enumerate(participants):
    row = activity[row_index]
    peak = int(np.max(row))
    shades = [('█' if value / peak >= .75 else '▒' if value / peak >= .50 else '░') if peak and value else '.' for value in row]
    print(f'  {person:<8} ' + ' '.join(shades))
print('\n  TOP GROUP WORDS')
for word, count in sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f'  {word:<12}: {count}')
print('\n  RESPONSE PATTERNS')
if average_response:
    fastest = min(average_response, key=average_response.get)
    slowest = max(average_response, key=average_response.get)
    print(f'  Fastest replier: {fastest} ({average_response[fastest]:.1f} minutes average)')
    print(f'  Slowest replier: {slowest} ({average_response[slowest] / 60:.1f} hours average)')
print('\n  LONGEST SILENT STREAKS')
for person in sorted(participants, key=lambda name: longest_silent[name], reverse=True):
    print(f'  {person:<10}: {longest_silent[person]} days')
print('\n  PERSONALITY ARCHETYPES')
for person in participants:
    print(f'  {person:<10} -> {archetypes[person][0]}')
print('\n' + '=' * 68)
print('  Generated by GroupDNA | Python + NumPy')
print('=' * 68)

## Reflection
**Hardest part:** [Write about the parser, timestamps, metrics, or another challenge you faced.]

**What I would do differently:** [Describe a change you would make with more time.]

**My archetype on a personal chat (optional):** [Add only if you choose to run this on a chat you have permission to analyze. Do not submit or publish the private chat export.]

## Run instructions
1. Install NumPy if it is not already available (`pip install numpy`).
2. Place `hostel_bois.txt` beside this notebook, or upload it into the Colab session.
3. Run all cells from top to bottom. If your file is in a different location, update `FILE_PATH` in Feature 1.
4. The assignment submission requires a single named `.ipynb` notebook and shareable Colab, GitHub, and LinkedIn links. Add your identity details above before submitting. The brief says to disclose any AI-assisted cell; the disclosure is included here.